# 01 — Auto-refresh free FX data and train

Academic research only; not financial advice. This notebook runs `refresh_dataset` to download any missing monthly partitions from your free provider, concatenates them into a validated training panel, and runs the research pipeline. Re-running it only downloads months that are missing or incomplete.

Prereqs: run `00_BOOTSTRAP_COLAB.ipynb` first, and store your API key in Colab `userdata` (see the bootstrap notebook).


In [ ]:
import os, sys
PROJECT = '/content/omega_worldclass_engine'
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)


In [ ]:
# Pull latest code, then load provider secrets from Colab userdata (presence only).
!git -C {PROJECT} pull --quiet
from omega.secrets import load_platform_secrets
print(load_platform_secrets())


In [ ]:
import json
from omega.cloud_config import load_cloud_config
from omega.acquisition import refresh_dataset, load_dataset
from omega.config import load_config
from omega.pipeline import run_pipeline

# Edit this config path to switch provider (cloud_twelvedata.yaml / cloud_polygon.yaml).
CLOUD_CONFIG = 'config/cloud_twelvedata.yaml'
# Edit these boundaries to your window. Use a month-aligned end like 2026-09-01T00:00:00Z.
START = '2026-01-01T00:00:00Z'
END = '2026-09-01T00:00:00Z'


## Step 1 — Refresh missing months

Set `explicit_terms_accepted: true` in `config/cloud_twelvedata.yaml` after reviewing the provider's terms, then run the refresh. `--max-partitions` bounds how many months a single run downloads.


In [ ]:
cloud = load_cloud_config(CLOUD_CONFIG)
refresh = refresh_dataset(
    cloud, PROJECT, START, END,
    accept_provider_terms=True,
    max_partitions=12,
)
print('fetched partitions:', refresh['fetched_partitions'])
print('already present:', refresh['already_present'])
print('integrity:', refresh['integrity']['total_rows'], 'rows,',
      refresh['integrity']['duplicate_timestamp_count'], 'duplicates')


## Step 2 — Load the validated panel

`load_dataset` fails closed if any requested month is missing or incomplete, so a partial dataset can never silently enter training.


In [ ]:
panel = load_dataset(cloud, PROJECT, START, END)
print('panel rows:', len(panel))
print('range:', panel.timestamp.min(), '->', panel.timestamp.max())
panel.head()


## Step 3 — Run the research pipeline

Metrics and predictions are written under `artifacts/colab_run` with per-stage checkpointing, so an interrupted run resumes instead of restarting.


In [ ]:
research = load_config('config.yaml')
research['data_source'] = cloud['data_source']
metrics = run_pipeline(panel, research, 'artifacts/colab_run')
display(metrics.groupby(['label', 'model'])[['brier', 'average_precision', 'ece']].mean().round(4))
